# 3 · Schema mapping for a migration

Migrating to a lakehouse means mapping hundreds of legacy columns to a target model. Today that's a spreadsheet and naming guesswork: `KLT_NM_VOL` → `customer.full_name`?

**The pattern:** code shortlists the plausible targets → Jev picks one with a **Choice** (a probability per option) → code pre-fills the confident, conflict-free mappings and sends the rest to the consultant.

In [ ]:
import json

import pandas as pd

from jevdemo import schema
from jevdemo.judge import MODEL, Judge, LiveUnavailable, NotRecorded, request_body

judge = Judge()  # JEV_MODE=live for live calls; default replays recordings
print(f"mode: {judge.mode} · model: {MODEL}")

## Source and target
25 columns from a synthetic Dutch legacy CRM, and a 30-column target model. `schema_gold.csv` is the mapping a consultant would have signed off.

In [ ]:
sources, targets, gold = (
    schema.load_sources(),
    schema.load_targets(),
    schema.load_gold(),
)
display(pd.DataFrame(sources).head(10), pd.DataFrame(targets).head(10))

## Code step: shortlist
The consultant declares which target tables a source table feeds; code keeps the type-compatible columns and always adds `none`. A Choice takes up to 255 options (measured), so the shortlist is about relevance, not size.

In [ ]:
shortlists = {s["source_col"]: schema.shortlist(s, targets) for s in sources}
print(
    f"options per column: {min(map(len, shortlists.values()))}–{max(map(len, shortlists.values()))}"
)
print(
    f"shortlist recall: {sum(gold[c] in o for c, o in shortlists.items())} of {len(gold)} columns have the right target in their list"
)

## What Jev receives

In [ ]:
source = next(s for s in sources if s["source_col"] == "KLANT.KLT_NM_VOL")
print(
    json.dumps(
        request_body(*schema.mapping_request(source, targets)),
        indent=2,
        ensure_ascii=False,
    )
)

## Jev maps every column

In [ ]:
answers = judge.ask_many([schema.mapping_request(s, targets) for s in sources])
mappings = [
    {**schema.to_mapping(s, a.response), "shortlist": shortlists[s["source_col"]]}
    for s, a in zip(sources, answers)
]

## Pre-fill or review
Pre-filled when the chosen option has `p ≥ 0.80` and no other column claims the same 1:1 target. Everything else goes to the consultant with a reason.

In [ ]:
decided = schema.decide(mappings)
table = pd.DataFrame(decided)[["source_col", "target_col", "p", "decision", "reason"]]
table["gold"] = table["source_col"].map(gold)
table["correct"] = table["target_col"] == table["gold"]
table.round(2)

## Against the signed-off mapping

In [ ]:
pd.Series(schema.evaluate(decided, gold)).round(2)

## Where it went wrong
Wrong answers that went to review cost the consultant a minute. A wrong answer that was pre-filled is the expensive kind; this is why a person approves the file.

In [ ]:
table[~table["correct"]].round(2)

## The deliverable: dbt `sources.yml`
Only pre-filled mappings; the review rows go to the consultant's list.

In [ ]:
print(schema.sources_yml(decided, source_name="legacy_crm"))

## What we tried

- **As specified:** `none` as one of the Choice options. **76% top-1, 60% of columns pre-filled correctly.**
- Adding generic context to the question ("names are abbreviated Dutch; cents versus euros is still the same data"): 64% pre-filled correctly.
- Replacing the `none` option with a companion Noul, as the Choice docs suggest: 36%, because the Noul was under-confident on most columns.

The spec's target was ≥ 70% pre-filled correctly. On this small synthetic set, we didn't reach it. The errors are mostly Dutch abbreviations without comments (`LVR`, `VRZ`, `BTW`), and a consultant's abbreviation list is the obvious next lever.

## Your turn
Describe a column. The preset is recorded; a new column needs `JEV_MODE=live`.

In [ ]:
column = schema.LIVE_PRESET  # or edit name, type, comment, samples

try:
    answer = judge.ask(*schema.mapping_request(column, targets))
    m = schema.to_mapping(column, answer.response)
    print(
        f"{m['source_col']} → {m['target_col']}  p={m['p']:.2f}  ({answer.source}, {answer.latency_ms} ms)"
    )
except (NotRecorded, LiveUnavailable) as e:
    print(e)